# Prédiction du temps de livraison — Food Delivery

**Objectif métier** : prédire `delivery_time_min` (temps de livraison, en minutes) à partir des caractéristiques de la commande, du livreur, du trafic et de la météo.

**Dataset** : `food-delivery.csv` — 45593 lignes, 20 colonnes brutes.

## Sommaire
1. Chargement et nettoyage
2. Exploration (EDA)
3. Transformation et modélisation
4. Optimisation et évaluation

## 1. Chargement et nettoyage

### 1.1 Import des librairies et chargement des données

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/food-delivery.csv")
# df.columns.tolist()
df.shape

(45593, 20)

### 1.2 Suppression des colonnes sans valeur prédictive

- `ID` : identifiant unique de la commande, aucune valeur prédictive.
- `Delivery_person_ID` : identifiant du livreur (1320 valeurs uniques).

In [2]:
df = df.drop(columns=['ID', 'Delivery_person_ID'])
df.shape

(45593, 18)

### 1.3 Renommage des colonnes

In [3]:
df = df.rename(columns={
    'Delivery_person_Age': 'delivery_person_age',
    'Delivery_person_Ratings': 'delivery_person_ratings',
    'Restaurant_latitude': 'restaurant_lat',
    'Restaurant_longitude': 'restaurant_lon',
    'Delivery_location_latitude': 'delivery_lat',
    'Delivery_location_longitude': 'delivery_lon',
    'Order_Date': 'order_date',
    'Time_Orderd': 'time_ordered',
    'Time_Order_picked': 'time_picked',
    'Weatherconditions': 'weather',
    'Road_traffic_density': 'traffic',
    'Vehicle_condition': 'vehicle_condition',
    'Type_of_order': 'order_type',
    'Type_of_vehicle': 'vehicle_type',
    'Festival': 'festival',
    'City': 'city',
    'Time_taken(min)': 'delivery_time_min',
})

df.columns.tolist()

['delivery_person_age',
 'delivery_person_ratings',
 'restaurant_lat',
 'restaurant_lon',
 'delivery_lat',
 'delivery_lon',
 'order_date',
 'time_ordered',
 'time_picked',
 'weather',
 'traffic',
 'vehicle_condition',
 'order_type',
 'vehicle_type',
 'multiple_deliveries',
 'festival',
 'city',
 'delivery_time_min']

### 1.4 Nettoyage des espaces parasites


In [4]:
columns_str = df.select_dtypes(include='str').columns
# print(columns_str)

for col in columns_str:
    df[col] = df[col].str.strip()

# for col in columns_str:
#     print(df[col].unique())
#     print('----------------')
df['traffic'].unique()

<StringArray>
['High', 'Jam', 'Low', 'Medium', 'NaN']
Length: 5, dtype: str

### 1.5 Conversion des colonnes numériques

`delivery_person_age`, `delivery_person_ratings`, `multiple_deliveries` sont en texte. Aucune valeur mal formée détectée (vérifié : pas de virgule décimale). `pd.to_numeric(errors='coerce')` convertit directement les chaînes numériques et transforme `"NaN"` en `np.nan` au passage.

In [5]:
# for col in ['delivery_person_age', 'delivery_person_ratings', 'multiple_deliveries']:
#     suspects = df[col][df[col].astype(str).str.contains(',')]
#     print(col, ':', suspects.unique())
numeric_cols = ['delivery_person_age', 'delivery_person_ratings', 'multiple_deliveries']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# print(df.info())
df[numeric_cols].isna().sum()



delivery_person_age        1854
delivery_person_ratings    1908
multiple_deliveries         993
dtype: int64

### 1.6 Traitement de la valeur aberrante dans `delivery_person_ratings`


In [6]:
df.loc[df['delivery_person_ratings'] == 6, 'delivery_person_ratings'] = np.nan
print((df['delivery_person_ratings'] == 6).sum())
print(df['delivery_person_ratings'].isna().sum())
print(df['delivery_person_ratings'].max())



0
1961
5.0


### 1.7 Correction des coordonnées GPS négatives


In [7]:
# df.columns.tolist()
df['restaurant_lat'] = df['restaurant_lat'].abs()
df['restaurant_lon'] = df['restaurant_lon'].abs()
print((df['restaurant_lat'] < 0).sum())
print((df['restaurant_lon'] < 0).sum())


0
0


### 1.8 Traitement des coordonnées placeholder


In [8]:
df.loc[(df['restaurant_lat'] == 0) & (df['restaurant_lon'] == 0), ['restaurant_lat', 'restaurant_lon']] = np.nan
df.loc[(df['delivery_lat'] == 0.01) & (df['delivery_lon'] == 0.01), ['delivery_lat', 'delivery_lon']] = np.nan

print("Restaurant coords manquantes :", df['restaurant_lat'].isna().sum())
print("Delivery coords manquantes :", df['delivery_lat'].isna().sum())



Restaurant coords manquantes : 3640
Delivery coords manquantes : 327


### 1.9 Extraction de la cible et nettoyage de la météo


In [9]:
# df['delivery_time_min'] = df['delivery_time_min'].str.extract(r'(\d+)').astype(int)
df['weather'] = df['weather'].str.replace('conditions ', '', regex=False)
# df['delivery_time_min'].head()
df['weather'].unique()
print(df['weather'].isna().sum())


0


### 1.9 Correction du "NaN" textuel restant dans `weather`


In [10]:
df['weather'] = df['weather'].replace('NaN', np.nan, regex=False)
df['weather'].isna().sum()

np.int64(616)

### 1.10 Conversion des dates et heures


In [12]:
df['order_date'] = pd.to_datetime(df['order_date'], format='%d-%m-%Y', errors='coerce')

time_ordered = pd.to_datetime(df['time_ordered'], format='%H:%M:%S', errors='coerce')
time_picked = pd.to_datetime(df['time_picked'], format='%H:%M:%S', errors='coerce')

mask_minuit = time_picked < time_ordered
time_picked = time_picked.where(~mask_minuit, time_picked + pd.Timedelta(days=1))

df['time_ordered'] = time_ordered
df['time_picked'] = time_picked

print(df[['order_date', 'time_ordered', 'time_picked']].dtypes)
print((df['time_picked'] < df['time_ordered']).sum())

order_date      datetime64[us]
time_ordered    datetime64[us]
time_picked     datetime64[us]
dtype: object
0


### 1.12 Imputation des valeurs manquantes

Conformément à la consigne du brief, l'imputation est effectuée directement à cette étape, sur l'ensemble du dataset :
- **Variables numériques** (`delivery_person_age`, `delivery_person_ratings`) : imputation par la **médiane**, robuste aux valeurs extrêmes.
- **Variables catégorielles à faible cardinalité et déséquilibrées** (`multiple_deliveries`, `festival`) : imputation par le **mode** (valeur la plus fréquente).

Les colonnes `traffic`, `city`, ainsi que les coordonnées GPS et `time_ordered`, sont traitées séparément à l'étape d'exploration (EDA), où l'analyse des fréquences et corrélations permettra un choix plus informé.

In [ ]:
# Médiane pour les numériques
for col in ['delivery_person_age', 'delivery_person_ratings']:
    median = df[col].median()
    df[col] = df[col].fillna(median)
    print(f"{col} est impute avec: {median}")

    # Mode pour les catégorielles simples
for col in ['multiple_deliveries', 'festival']:
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)
    print(f"{col} : imputé avec le mode = {mode_val}")

print(df.isna().sum())

delivery_person_age est impute avec: 30.0
delivery_person_ratings est impute avec: 4.7
multiple_deliveries : imputé avec le mode = 1.0
festival : imputé avec le mode = No

Manquants restants :
delivery_person_age           0
delivery_person_ratings       0
restaurant_lat             3640
restaurant_lon             3640
delivery_lat                327
delivery_lon                327
order_date                    0
time_ordered               1731
time_picked                   0
weather                     616
traffic                       0
vehicle_condition             0
order_type                    0
vehicle_type                  0
multiple_deliveries           0
festival                      0
city                          0
delivery_time_min             0
dtype: int64


### 1.13 Correction du "NaN" textuel dans `traffic` et `city`


In [16]:
df['traffic'] = df['traffic'].replace('NaN', np.nan)
df['city'] = df['city'].replace('NaN', np.nan)

print(df[['traffic', 'city']].isna().sum())

traffic     601
city       1200
dtype: int64


In [ ]:
print("Restaurant_lat == 0 :", (df['restaurant_lat'] == 0).sum())
print("Restaurant_lon == 0 :", (df['restaurant_lon'] == 0).sum())
print("Les deux à 0 en même temps :", ((df['restaurant_lat'] == 0) & (df['restaurant_lon'] == 0)).sum())

# Comparaison avec delivery_lat/lon, qui n'avaient pas de négatif
print("Delivery_lat min :", df['delivery_lat'].min())
print("Delivery_lon min :", df['delivery_lon'].min())

print("Delivery_lat == 0.01 :", (df['delivery_lat'] == 0.01).sum())
print("Delivery_lon == 0.01 :", (df['delivery_lon'] == 0.01).sum())
print("Les deux à 0.01 en même temps :", ((df['delivery_lat'] == 0.01) & (df['delivery_lon'] == 0.01)).sum())

# Est-ce que les lignes à restaurant (0,0) sont les mêmes que celles à delivery (0.01, 0.01) ?
mask_resto_zero = (df['restaurant_lat'] == 0) & (df['restaurant_lon'] == 0)
mask_deliv_floor = (df['delivery_lat'] == 0.01) & (df['delivery_lon'] == 0.01)
print("Chevauchement des deux problèmes :", (mask_resto_zero & mask_deliv_floor).sum())

Restaurant_lat == 0 : 3640
Restaurant_lon == 0 : 3640
Les deux à 0 en même temps : 3640
Delivery_lat min : 0.01
Delivery_lon min : 0.01
Delivery_lat == 0.01 : 327
Delivery_lon == 0.01 : 327
Les deux à 0.01 en même temps : 327
Chevauchement des deux problèmes : 327
